# ECOSTRESS ET (ECO_L3T_JET v002) — Ogallala ROI, native 70 m, one CF file per MGRS tile
`earthaccess` catalogue search → 10 parallel downloads → corruption check → dask-filtered ROI subset →
CF NetCDF per tile, hourly Drive backup, resumable from the Drive `.nc`.

**Data:** ECOSTRESS Tiled Evapotranspiration L3 Global 70 m V002 (LP DAAC,
DOI 10.5067/ECOSTRESS/ECO_L3T_JET.002), 2018-07 → present, daytime ISS overpasses, one
GeoTIFF per layer per MGRS tile (1568 × 1568 px, 70 m, UTM). Layers extracted here:
`ETdaily` (float32, NaN = no data, mm/day) plus the `cloud` and `water` flags (uint8, 0/1, no
nodata value declared). v002 is the only version with the whole record (v003 covers 2025-26
only and uses different layer names). In the source, cloudy pixels already hold NaN in
`ETdaily`; water pixels can hold a value.

**No-manipulation design**
- Every tile stays on its own native UTM grid. Nothing is resampled, mosaicked, scaled, clipped
  or cast: each layer is read in its source dtype and written bit-for-bit.
- The only change is that pixels outside the aquifer polygon become `_FillValue`
  (one vectorised `np.where` with a boolean mask built once per tile).
- `cloud` and `water` are stored as their own variables and are **not** applied to `ETdaily`.
- No `valid_range` attribute is written (readers would silently mask on it).
- The first records of every tile (all of them in `TEST_MODE`) are re-read from the `.nc` and
  compared bitwise with the source GeoTIFF.

**What is never downloaded:** tiles with less than `MIN_TILE_ROI_FRAC` (5 %) of their area inside
the polygon. The catalogue carries no per-granule cloud or coverage figure, so whether an
acquisition actually holds data can only be checked after download; `should_write()` in Cell 7
decides whether it is written.

**Resume:** each tile file stores `resume_after` (last acquisition handled) and
`processing_status`. After a disconnect just *Run all*: finished tiles are skipped, the tile in
progress is restored from Drive and downloading restarts after its `resume_after`.

**Before the first run**
1. Colab → key icon (Secrets) → add `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` and allow
   notebook access. Without them the login falls back to a typed prompt, which blocks an
   unattended restart.
2. Leave `TEST_MODE = True` for the first run. It processes 2 tiles × 40 acquisitions (one tile
   mostly inside the aquifer, one on its edge) into `.../_test/`, checks every record bitwise and
   prints time by stage plus projected runtime and Drive size for the whole mission.
3. Set `TEST_MODE = False` and *Run all* for the full run. `ONLY_TILES` restricts a run to named
   tiles, which lets several Colab sessions work on different tiles at the same time.

**Outputs** (`MyDrive/MSUGWB/ECOSTRESS_ECO_L3T_JET_v002/`): `ECO_L3T_JET_v002_<tile>_Ogallala.nc`
per tile, the cached catalogue manifest, and JSON-lines logs (`corrupted_files`, `failed_granules`,
`skipped_granules`, `tile_summary`).

In [ ]:
# ═══════════════ CELL 1: install (pinned) ═══════════════
# fsspec/s3fs/gcsfs are pinned to 2025.3.0: the newer fsspec that pip pulls in by default
# has produced truncated earthaccess downloads in this project (see OCO SIF.ipynb, Cell 2).
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q earthaccess rasterio netCDF4 geopandas shapely pyproj "dask[distributed]" psutil "fsspec==2025.3.0" "s3fs==2025.3.0" "gcsfs==2025.3.0" 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> short end-to-end rehearsal into .../_test/ ; False -> whole mission
TEST_TILES    = 2          # tiles in the rehearsal, spread over the range of aquifer overlap
TEST_GRANULES = 40         # acquisitions per tile in the rehearsal
TEST_TEMPORAL = ("2022-06-01", "2022-09-01")   # catalogue window searched in the rehearsal

SHORT_NAME     = "ECO_L3T_JET"
VERSION        = "002"                 # 002 = whole record; 003 covers 2025-26 only
SCIENCE_LAYERS = ["ETdaily"]           # written bit-for-bit
QA_LAYERS      = ["cloud", "water"]    # stored as their own variables, never applied
MISSION_START  = "2018-07-09"
END_DATE       = None                  # None -> now (UTC); or "YYYY-MM-DD"

MIN_TILE_ROI_FRAC = 0.05   # tiles with a smaller share of their area inside the polygon are never downloaded
MIN_VALID_FRAC    = 0.05   # available to should_write() in Cell 7 (share of in-polygon pixels with data)
SKIP_GRANULES     = []     # granule names to treat as permanently unavailable (see the stall message)
ONLY_TILES        = None   # None -> every tile; or e.g. ["14SKF", "13TDG"] to process just these tiles
                           # (re-do one tile, or split the mission across several Colab sessions)

N_DOWNLOAD_THREADS = 10            # files downloaded at a time
PRIMARY_DOWNLOADER = "earthaccess" # "earthaccess" -> earthaccess.download(); "https" -> direct session only
DASK_WORKERS       = 6             # worker processes for verification and filtering
WORKER_MEMORY_GB   = 2.0           # hard memory limit per dask worker
GROUP_GRANULES     = 48            # acquisitions downloaded, verified and filtered per cycle
MAX_INFLIGHT       = 2 * DASK_WORKERS  # filter results allowed in memory at once (bounds RAM)
SYNC_EVERY         = 12            # records between flushes of the .nc to disk

COMPRESS_LEVEL = 1                 # zlib level of the CF .nc (lossless)
CHUNK_YX       = 512               # HDF5 chunk edge in y and x
WRITE_LATLON   = True              # also store 2-D lat/lon auxiliary coordinates

MIN_FREE_GB       = 20.0           # scratch space that must stay free at all times
DISK_CHECK_S      = 30             # watchdog sampling interval (seconds)
DISK_MAX_WAIT_S   = 2 * 3600       # give up (cleanly, resumable) if space never comes back
BACKUP_INTERVAL_S = 3600           # copy the tile in progress to Drive this often
MAX_RETRIES       = 4              # extra download attempts per file
STALL_MINUTES     = 20             # stop cleanly after this long with no progress at all
BITEXACT_CHECKS   = 3              # records per tile re-read and compared with the source (all in TEST_MODE)

MANIFEST_MAX_AGE_DAYS = 7          # refresh the cached catalogue listing when older than this
REBUILD_MANIFEST      = False      # True -> ignore the cached listing and search again
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, Earthdata login ═══════════════
import re, gc, sys, json, gzip, glob, time, math, shutil, hashlib, logging, threading, inspect, warnings
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed as cf_as_completed

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import rasterio
import rasterio.features
from rasterio.windows import Window
from shapely.geometry import Polygon, box
from shapely.geometry.polygon import orient
import shapely
import earthaccess
import dask
from dask.distributed import Client, LocalCluster, as_completed

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a
# no-op there and INFO progress lines would never be shown.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("ecostress")
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")   # DataGranule.size() notice
for _n in ("distributed", "urllib3.connectionpool", "rasterio", "pyogrio", "fiona"):
    logging.getLogger(_n).setLevel(logging.WARNING)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL   = "Ogallala"
PRODUCT_TAG = f"{SHORT_NAME}_v{VERSION}"
LAYERS      = list(SCIENCE_LAYERS) + list(QA_LAYERS)
PRIMARY     = SCIENCE_LAYERS[0]            # the layer the write/skip rule looks at

_SUFFIX    = "_test" if TEST_MODE else ""  # the rehearsal never touches real outputs
DRIVE_ROOT = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT  = f"{DRIVE_ROOT}/ECOSTRESS_{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")
SCRATCH    = "/content/eco_scratch" + _SUFFIX
RAW_DIR    = f"{SCRATCH}/raw"
NC_DIR     = f"{SCRATCH}/nc"
DASK_TMP   = f"{SCRATCH}/dask"
DISK_PATH  = "/content"                    # same filesystem as scratch AND Drive's local upload cache

# bare file names; directories are always joined explicitly (never join two absolute paths)
MANIFEST_NAME = f"manifest_{PRODUCT_TAG}.json.gz"
LOG_CORRUPT   = "corrupted_files.jsonl"
LOG_FAILED    = "failed_granules.jsonl"
LOG_SKIPPED   = "skipped_granules.jsonl"
LOG_TILES     = "tile_summary.jsonl"

for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, DASK_TMP):
    os.makedirs(_d, exist_ok=True)

# ---------- CF constants ----------
TIME_UNITS    = "seconds since 2000-01-01 00:00:00"   # CF: no time zone given means UTC
TIME_CALENDAR = "proleptic_gregorian"
EQUAL_AREA_CRS = "EPSG:5070"                          # CONUS Albers, for area fractions only
PRODUCT_DOI   = f"https://doi.org/10.5067/ECOSTRESS/{SHORT_NAME}.{VERSION}"

# Descriptive metadata only. Units follow the ECO_L3T_JET v002 user guide; the GeoTIFF's own
# tags are copied verbatim into each variable's `source_tags` attribute.
LAYER_META = {
    "ETdaily":           dict(long_name="Daily evapotranspiration (ETdaily layer of ECO_L3T_JET)", units="mm day-1"),
    "ETinstUncertainty": dict(long_name="Uncertainty of instantaneous evapotranspiration", units="W m-2"),
    "PTJPLSMinst":       dict(long_name="Instantaneous latent heat flux, PT-JPL-SM", units="W m-2"),
    "STICinst":          dict(long_name="Instantaneous latent heat flux, STIC", units="W m-2"),
    "MOD16inst":         dict(long_name="Instantaneous latent heat flux, MOD16", units="W m-2"),
    "BESSinst":          dict(long_name="Instantaneous latent heat flux, BESS", units="W m-2"),
    "cloud":             dict(long_name="Cloud mask (1 = cloud)", units="1",
                              flag_values=(0, 1), flag_meanings="clear cloud"),
    "water":             dict(long_name="Water mask (1 = water)", units="1",
                              flag_values=(0, 1), flag_meanings="not_water water"),
}


class PipelineStop(RuntimeError):
    """Clean stop: everything written so far is safe, re-running resumes."""


class AuthError(PipelineStop):
    """Earthdata refused the credentials."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_iso(s):
    return datetime.strptime(s, "%Y-%m-%dT%H:%M:%SZ")


def parse_day(s):
    return datetime.strptime(s, "%Y-%m-%d")


def earthdata_login():
    """Log in without typing when Colab Secrets hold the credentials (needed for an
    unattended restart); otherwise fall back to the interactive prompt."""
    try:
        from google.colab import userdata
        for key in ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD"):
            try:
                val = userdata.get(key)
                if val:
                    os.environ[key] = val
            except Exception:
                pass
    except Exception:
        pass
    if not (os.environ.get("EARTHDATA_TOKEN") or (os.environ.get("EARTHDATA_USERNAME")
                                                  and os.environ.get("EARTHDATA_PASSWORD"))):
        log.info("No Colab Secret / environment credentials - earthaccess will try ~/.netrc, then prompt.")
    auth = earthaccess.login()             # order: environment -> ~/.netrc -> interactive prompt
    if not getattr(auth, "authenticated", False):
        raise AuthError("Earthdata login failed - check EARTHDATA_USERNAME / EARTHDATA_PASSWORD.")
    log.info("Earthdata login OK")
    return auth

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs, stall watchdog ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
LOW_DISK    = threading.Event()          # set by the watchdog; downloads are withheld while set
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def cleanup_scratch():
    """Delete only disposable leftovers (interrupted .part downloads)."""
    n = 0
    for f in glob.glob(os.path.join(RAW_DIR, "**", "*.part"), recursive=True):
        try:
            os.remove(f)
            n += 1
        except OSError:
            pass
    return n


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds one download
    group and one tile file, so a shortfall normally means Drive's local upload cache is
    still draining: wait for it rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - cleaning up and waiting for Drive to upload",
                        free, need, context)
            cleanup_scratch()
            continue
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


class DiskWatchdog(threading.Thread):
    """Samples free space in the background and raises/clears the LOW_DISK flag."""

    def __init__(self):
        super().__init__(daemon=True, name="disk-watchdog")
        self._halt = threading.Event()

    def run(self):
        beat = 0
        while not self._halt.is_set():
            try:
                free = free_gb()
                if free < MIN_FREE_GB:
                    if not LOW_DISK.is_set():
                        log.warning("WATCHDOG: %.1f GB free < %.1f GB floor - new downloads withheld",
                                    free, MIN_FREE_GB)
                    LOW_DISK.set()
                else:
                    LOW_DISK.clear()
                    beat += 1
                    if beat % max(1, 1800 // DISK_CHECK_S) == 1:
                        log.info("WATCHDOG: disk healthy (%.1f GB free)", free)
            except Exception as e:
                log.warning("WATCHDOG: %s", e)
            self._halt.wait(DISK_CHECK_S)

    def stop(self):
        self._halt.set()


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))


# ---------- stop after a long stretch with no progress at all ----------
class StallWatchdog:
    def __init__(self, limit_min):
        self.limit = limit_min * 60.0
        self.since = None
        self.n = 0

    def progress(self):
        self.since, self.n = None, 0

    def failure(self):
        """Record a cycle without progress; True once the stall has lasted too long."""
        now = time.time()
        if self.since is None:
            self.since = now
        self.n += 1
        return (now - self.since) > self.limit


STALL = StallWatchdog(STALL_MINUTES)

In [ ]:
# ═══════════════ CELL 5: ROI polygon, catalogue manifest, tile selection ═══════════════
ROUGH_BUFFER_DEG   = 0.15    # buffer > simplify tolerance, so the rough ring always contains the polygon
ROUGH_SIMPLIFY_DEG = 0.10


def load_roi(shp_path):
    """Exact polygon (EPSG:4326, unsimplified: used for the 70 m masks and the 5 % rule) and
    a rough single-ring version (used only as the spatial filter of the catalogue search)."""
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        raise ValueError(f"{shp_path} has no CRS")
    g4 = gdf.to_crs(4326).geometry
    exact = g4.union_all() if hasattr(g4, "union_all") else g4.unary_union
    if not exact.is_valid:
        exact = exact.buffer(0)
    rough = exact.buffer(ROUGH_BUFFER_DEG).simplify(ROUGH_SIMPLIFY_DEG, preserve_topology=True)
    if rough.geom_type != "Polygon":
        rough = rough.convex_hull
    ring = orient(Polygon(rough.exterior), sign=1.0)          # CMR wants one counter-clockwise ring
    coords = [(round(x, 4), round(y, 4)) for x, y in ring.exterior.coords]
    return exact, coords


ROI_EXACT, ROI_RING = load_roi(SHP_PATH)
ROI_EA = gpd.GeoSeries([ROI_EXACT], crs=4326).to_crs(EQUAL_AREA_CRS).iloc[0]
log.info("ROI bounds %s | area %.0f km2 | search ring %d vertices",
         [round(v, 3) for v in ROI_EXACT.bounds], ROI_EA.area / 1e6, len(ROI_RING))


def tile_roi_fraction(bbox):
    """Share of a tile's catalogue footprint (W, S, E, N) that lies inside the exact polygon,
    measured in an equal-area projection. Needs no download."""
    rect = shapely.segmentize(box(*bbox), 0.02)
    rect_ea = gpd.GeoSeries([rect], crs=4326).to_crs(EQUAL_AREA_CRS).iloc[0]
    return float(rect_ea.intersection(ROI_EA).area / rect_ea.area)


# ECOv002_L3T_JET_22615_014_14TLK_20220702T144842_0712_01   (v002: build + iteration)
# ECOv003_L3T_JET_44814_012_14SQE_20260601T003431_02        (v003: iteration only)
UR_RE = re.compile(r"^ECOv(?P<ver>\d{3})_(?P<prod>L\d[A-Z]+_[A-Z0-9]+)_(?P<orbit>\d+)_(?P<scene>\d+)"
                   r"_(?P<tile>\d{2}[A-Z]{3})_(?P<dt>\d{8}T\d{6})_(?P<rev>\d+(?:_\d+)*)$")


def parse_ur(ur):
    m = UR_RE.match(ur)
    if not m:
        return None
    d = m.groupdict()
    try:
        t = datetime.strptime(d["dt"], "%Y%m%dT%H%M%S")      # acquisition start, UTC
    except ValueError:
        return None
    return dict(tile=d["tile"], t=t, orbit=int(d["orbit"]), scene=int(d["scene"]), rev=d["rev"])


def rev_key(rev):
    return tuple(int(p) for p in rev.split("_"))


def _umm_bbox(umm):
    geom = umm.get("SpatialExtent", {}).get("HorizontalSpatialDomain", {}).get("Geometry", {})
    if geom.get("BoundingRectangles"):
        r = geom["BoundingRectangles"][0]
        return [r["WestBoundingCoordinate"], r["SouthBoundingCoordinate"],
                r["EastBoundingCoordinate"], r["NorthBoundingCoordinate"]]
    if geom.get("GPolygons"):
        pts = geom["GPolygons"][0]["Boundary"]["Points"]
        xs, ys = [p["Longitude"] for p in pts], [p["Latitude"] for p in pts]
        return [min(xs), min(ys), max(xs), max(ys)]
    return None


def _umm_mb(umm):
    total = 0.0
    for a in umm.get("DataGranule", {}).get("ArchiveAndDistributionInformation", []):
        if a.get("SizeInBytes"):
            total += a["SizeInBytes"] / 1e6
        elif a.get("Size") is not None:
            total += float(a["Size"]) * {"KB": 1e-3, "MB": 1.0, "GB": 1e3}.get(a.get("SizeUnit", "MB"), 1.0)
    return round(total, 2)


def slim_record(g):
    """Reduce one earthaccess granule to the few fields the pipeline needs.
    Returns (record, None) or (None, reason)."""
    umm = g["umm"]
    ur = umm["GranuleUR"]
    if parse_ur(ur) is None:
        return None, "name not understood"
    try:
        links = g.data_links(access="external")
    except TypeError:
        links = g.data_links()
    urls = {}
    for u in links:
        u = u.split("?")[0]
        name = u.rsplit("/", 1)[-1]
        if u.startswith("http") and name.startswith(ur + "_") and name.endswith(".tif"):
            urls[name[len(ur) + 1:-4]] = u
    missing = [l for l in LAYERS if l not in urls]
    if missing:
        return None, f"layers not offered: {missing}"
    base = urls[LAYERS[0]].rsplit("/", 1)[0]
    rec = dict(ur=ur, base=base, mb=_umm_mb(umm), bbox=_umm_bbox(umm))
    if any(urls[l] != f"{base}/{ur}_{l}.tif" for l in LAYERS):
        rec["urls"] = {l: urls[l] for l in LAYERS}           # unusual layout: keep the links verbatim
    return rec, None


def month_windows(start, end):
    cur = start
    while cur < end:
        nxt = datetime(cur.year + (cur.month == 12), cur.month % 12 + 1, 1)
        yield cur, min(nxt, end)
        cur = nxt


def safe_search(start, end, retries=4):
    """One catalogue window. A failed window raises: silently returning nothing would
    leave a permanent hole in the manifest."""
    last = None
    for attempt in range(1, retries + 1):
        try:
            res = earthaccess.search_data(short_name=SHORT_NAME, version=VERSION, polygon=ROI_RING,
                                          temporal=(iso(start), iso(end - timedelta(seconds=1))), count=-1)
            return list(res) if res else []
        except Exception as e:
            last = e
            log.warning("Catalogue search %s..%s attempt %d/%d failed: %s",
                        start.date(), end.date(), attempt, retries, e)
            time.sleep(15 * attempt)
    raise PipelineStop(f"Catalogue search failed for {start.date()}..{end.date()}: {last}")


def search_into(recs, start, end):
    dropped = []
    for s, e in month_windows(start, end):
        res = safe_search(s, e)
        for g in res:
            rec, why = slim_record(g)
            if rec is None:
                dropped.append(dict(ur=g["umm"].get("GranuleUR", "?"), stage="catalogue", reason=why))
            else:
                recs[rec["ur"]] = rec
        log.info("  catalogue %s: %5d granules (total %d)", s.strftime("%Y-%m"), len(res), len(recs))
        del res
        gc.collect()
    if dropped:
        log.warning("%d catalogue records ignored (see %s)", len(dropped), LOG_SKIPPED)
        log_events(LOG_SKIPPED, dropped)
    return recs


def manifest_signature(start):
    key = json.dumps([SHORT_NAME, VERSION, LAYERS, ROI_RING, iso(start)])
    return hashlib.md5(key.encode()).hexdigest()


def load_cached_manifest():
    path = os.path.join(DRIVE_OUT, MANIFEST_NAME)
    if not os.path.exists(path):
        return None
    try:
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        log.warning("Cached manifest unreadable (%s) - searching again", e)
        return None


def save_manifest(recs, start, through):
    payload = dict(meta=dict(short_name=SHORT_NAME, version=VERSION, layers=LAYERS, start=iso(start),
                             listed_through=iso(through), signature=manifest_signature(start), n=len(recs)),
                   granules=list(recs.values()))
    local = os.path.join(NC_DIR, MANIFEST_NAME)
    with gzip.open(local, "wt", encoding="utf-8") as f:
        json.dump(payload, f)
    robust_drive_copy(local, os.path.join(DRIVE_OUT, MANIFEST_NAME))


def get_manifest():
    """Catalogue listing of every granule over the polygon, cached on Drive so that a
    restart skips the search. Only the tail is searched again when the cache is stale."""
    if TEST_MODE:
        start, end = parse_day(TEST_TEMPORAL[0]), parse_day(TEST_TEMPORAL[1])
    else:
        start = parse_day(MISSION_START)
        end = parse_day(END_DATE) if END_DATE else utcnow()
    cached = None if REBUILD_MANIFEST else load_cached_manifest()
    if cached and cached.get("meta", {}).get("signature") == manifest_signature(start):
        recs = {r["ur"]: r for r in cached["granules"]}
        through = parse_iso(cached["meta"]["listed_through"])
        fresh = through >= end or (utcnow() - through) < timedelta(days=MANIFEST_MAX_AGE_DAYS)
        if fresh:
            log.info("Manifest: %d granules from the Drive cache (listed through %s)", len(recs), through.date())
            return recs
        log.info("Manifest: cache is %d days old - searching the tail from %s",
                 (utcnow() - through).days, (through - timedelta(days=10)).date())
        search_into(recs, max(start, through - timedelta(days=10)), end)
    else:
        log.info("Manifest: searching %s v%s over the polygon, %s .. %s (one-time, a few minutes)",
                 SHORT_NAME, VERSION, start.date(), end.date())
        recs = search_into({}, start, end)
    if not recs:
        raise PipelineStop("The catalogue returned no granules - check SHORT_NAME / VERSION / dates.")
    save_manifest(recs, start, min(end, utcnow()))
    return recs


def build_tile_index(recs):
    """Group granules by MGRS tile, keep only the newest revision of each acquisition and
    attach each tile's share of area inside the polygon."""
    tiles, superseded = {}, 0
    for r in recs.values():
        p = parse_ur(r["ur"])
        g = dict(ur=r["ur"], t=p["t"], orbit=p["orbit"], scene=p["scene"], rev=p["rev"],
                 base=r["base"], mb=r.get("mb", 0.0), urls=r.get("urls"))
        T = tiles.setdefault(p["tile"], dict(bbox=None, by_t={}))
        if T["bbox"] is None and r.get("bbox"):
            T["bbox"] = r["bbox"]
        old = T["by_t"].get(p["t"])
        if old is not None:
            superseded += 1
            if rev_key(g["rev"]) <= rev_key(old["rev"]):
                continue
        T["by_t"][p["t"]] = g
    index = {}
    for tile, T in tiles.items():
        frac = tile_roi_fraction(T["bbox"]) if T["bbox"] else float("nan")
        index[tile] = dict(tile=tile, bbox=T["bbox"], roi_frac=frac,
                           granules=sorted(T["by_t"].values(), key=lambda g: g["t"]))
    if superseded:
        log.info("Tile index: %d older revisions of re-processed acquisitions ignored", superseded)
    return index


def select_tiles(index):
    """The pre-download filter: a tile with less than MIN_TILE_ROI_FRAC of its area inside
    the polygon is dropped here, so none of its files is ever requested."""
    keep, drop = [], []
    for tile in sorted(index):
        T = index[tile]
        (drop if T["roi_frac"] < MIN_TILE_ROI_FRAC else keep).append(T)   # NaN (no footprint) is kept
    n_keep = sum(len(T["granules"]) for T in keep)
    n_drop = sum(len(T["granules"]) for T in drop)
    log.info("Tiles: %d kept (%d granules), %d dropped below %.0f %% overlap (%d granules never downloaded)",
             len(keep), n_keep, len(drop), 100 * MIN_TILE_ROI_FRAC, n_drop)
    for T in drop:
        log.info("   dropped %s: %.2f %% inside the polygon, %d granules",
                 T["tile"], 100 * T["roi_frac"], len(T["granules"]))
    return keep


def granule_urls(g):
    return g["urls"] if g.get("urls") else {l: f"{g['base']}/{g['ur']}_{l}.tif" for l in LAYERS}


def granule_files(g, raw_dir):
    return {l: os.path.join(raw_dir, f"{g['ur']}_{l}.tif") for l in LAYERS}

In [ ]:
# ═══════════════ CELL 6: download (10 at a time) + corruption check after every download ═══════════════
_tls = threading.local()
_DL_KW = {"show_progress": False} if "show_progress" in inspect.signature(earthaccess.download).parameters else {}


def _session():
    """One authenticated HTTPS session per download thread."""
    s = getattr(_tls, "session", None)
    if s is None:
        s = earthaccess.get_requests_https_session()
        _tls.session = s
    return s


def https_fetch(url, dest):
    """Fetch one file over the authenticated session into dest.part, compare its size with
    the server's Content-Length, then rename. Returns 'ok' or 'missing' (HTTP 404)."""
    if LOW_DISK.is_set():
        raise IOError("disk floor reached - download withheld")
    tmp = dest + ".part"
    with _session().get(url, stream=True, timeout=(30, 300), allow_redirects=True) as r:
        if r.status_code in (401, 403):
            raise AuthError(f"HTTP {r.status_code} for {os.path.basename(dest)} - Earthdata session rejected")
        if r.status_code == 404:
            return "missing"
        r.raise_for_status()
        ctype = (r.headers.get("Content-Type") or "").lower()
        if "html" in ctype:
            raise IOError(f"server returned {ctype} (login redirect?), not data")
        expected = r.headers.get("Content-Length")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                if chunk:
                    f.write(chunk)
    got = os.path.getsize(tmp)
    if expected is not None and got != int(expected):
        os.remove(tmp)
        raise IOError(f"size mismatch: got {got}, server says {expected}")
    os.replace(tmp, dest)
    return "ok"


def fetch_files(pairs, raw_dir, primary):
    """Download (url, dest) pairs, N_DOWNLOAD_THREADS at a time.
    primary=True  : earthaccess.download() first, then the direct session for whatever is missing.
    primary=False : direct session only (used for re-fetches; it is size-checked and atomic).
    Returns {dest: 'ok' | 'missing' | 'error: ...'} for files that went through the direct session."""
    todo = [(u, d) for u, d in pairs if not os.path.exists(d)]
    out = {}
    if not todo:
        return out
    if primary and PRIMARY_DOWNLOADER == "earthaccess":
        try:
            earthaccess.download([u for u, _ in todo], local_path=raw_dir,
                                 threads=N_DOWNLOAD_THREADS, **_DL_KW)
        except Exception as e:
            log.warning("earthaccess.download raised (%s) - fetching the rest file by file", str(e)[:160])
        todo = [(u, d) for u, d in todo if not os.path.exists(d)]
    auth_error = None
    if todo:
        with ThreadPoolExecutor(max_workers=N_DOWNLOAD_THREADS) as pool:
            futs = {pool.submit(https_fetch, u, d): d for u, d in todo}
            for fut in cf_as_completed(futs):
                d = futs[fut]
                try:
                    out[d] = fut.result()
                except AuthError as e:
                    auth_error = e
                    out[d] = f"error: {e}"
                except Exception as e:
                    out[d] = f"error: {type(e).__name__}: {str(e)[:160]}"
    if auth_error is not None:
        raise auth_error
    return out


def verify_raster(path):
    """Integrity check of one downloaded GeoTIFF: size, TIFF magic bytes, not an HTML error
    page, opens, has a CRS, and EVERY block decodes (a truncated file passes a thumbnail
    read but not this). Pure, never raises; runs on a dask worker."""
    import os
    out = {"path": path, "ok": False, "reason": ""}
    try:
        size = os.path.getsize(path)
        if size < 128:
            out["reason"] = f"too small ({size} B)"
            return out
        with open(path, "rb") as fh:
            head = fh.read(512)
        if head[:4] not in (b"II*\x00", b"MM\x00*", b"II+\x00", b"MM\x00+"):
            html = b"<" in head[:64] or b"html" in head.lower()
            out["reason"] = "HTML/login page instead of data" if html else "bad TIFF magic bytes"
            return out
        import rasterio
        with rasterio.open(path) as src:
            if src.count < 1 or src.width < 1 or src.height < 1:
                out["reason"] = "empty dimensions"
                return out
            if src.crs is None:
                out["reason"] = "missing CRS"
                return out
            arr = src.read(1)
            tags = dict(src.tags())
            tags.update({f"band1:{k}": v for k, v in src.tags(1).items()})
            out.update(dtype=str(arr.dtype), nodata=src.nodata, crs_wkt=src.crs.to_wkt(),
                       transform=tuple(src.transform)[:6], shape=(src.height, src.width),
                       size=size, tags=tags)
        out["ok"] = True
    except Exception as e:
        out["reason"] = f"unreadable: {type(e).__name__}: {str(e)[:160]}"
    return out


def download_and_verify(granules, raw_dir, client):
    """Download every layer of the given granules, then check each file for corruption.
    Bad files are deleted and re-fetched one by one over the size-checked session.

    Returns (ok, permanent, transient):
      ok        [(granule, {layer: path}, {layer: profile})]  every layer present and verified
      permanent [(granule, reason)]  missing at source, or corrupt in >= 2 size-checked downloads
      transient [(granule, reason)]  could not be fetched (network / outage) - retry later
    """
    os.makedirs(raw_dir, exist_ok=True)
    files = {}
    for gi, g in enumerate(granules):
        urls, dests = granule_urls(g), granule_files(g, raw_dir)
        for l in LAYERS:
            files[dests[l]] = dict(url=urls[l], gi=gi, layer=l, state="need", why="not downloaded",
                                   corrupt_verified=0, profile=None)
    corrupt_log = []
    for attempt in range(MAX_RETRIES + 1):
        need = [d for d, f in files.items() if f["state"] == "need"]
        if not need:
            break
        if attempt:
            time.sleep(min(5 * 2 ** attempt, 60))
        with timed("download"):
            res = fetch_files([(files[d]["url"], d) for d in need], raw_dir, primary=(attempt == 0))
        present = []
        for d in need:
            if res.get(d) == "missing":
                files[d].update(state="missing", why="HTTP 404 at source")
            elif os.path.exists(d):
                present.append(d)
            else:
                files[d]["why"] = res.get(d, "not downloaded")
        # ---- corruption check at the end of the download (parallel, on the dask workers) ----
        t_check = time.time()
        futs = client.map(verify_raster, present, pure=False) if present else []
        n_bad = 0
        for d, fut in zip(present, futs):
            try:
                v = fut.result()
            except Exception as e:
                v = {"ok": False, "reason": f"verify task failed: {type(e).__name__}: {e}"}
            if v["ok"]:
                files[d].update(state="ok", profile=v)
                continue
            n_bad += 1
            size_checked = res.get(d) == "ok"       # came through the Content-Length-checked path
            files[d]["corrupt_verified"] += int(size_checked)
            files[d]["why"] = "corrupt: " + v["reason"]
            if files[d]["corrupt_verified"] >= 2:   # full-size download, twice, still undecodable
                files[d]["state"] = "corrupt"
            corrupt_log.append(dict(file=os.path.basename(d), attempt=attempt, reason=v["reason"],
                                    size_checked=size_checked))
            try:
                os.remove(d)
            except OSError:
                pass
        del futs
        TIMES["corruption check"] = TIMES.get("corruption check", 0.0) + time.time() - t_check
        if present:
            log.info("   corruption check (attempt %d): %d files, %d corrupt%s",
                     attempt, len(present), n_bad, " -> deleted, fetching again" if n_bad else "")
    log_events(LOG_CORRUPT, corrupt_log)

    ok, permanent, transient = [], [], []
    for gi, g in enumerate(granules):
        mine = {f["layer"]: (d, f) for d, f in files.items() if f["gi"] == gi}
        if all(f["state"] == "ok" for _, f in mine.values()):
            ok.append((g, {l: d for l, (d, _) in mine.items()}, {l: f["profile"] for l, (_, f) in mine.items()}))
            continue
        bad = [(l, f) for l, (_, f) in mine.items() if f["state"] != "ok"]
        perm = [(l, f) for l, f in bad if f["state"] in ("missing", "corrupt")]
        why = "; ".join(f"{l}: {f['why']}" for l, f in bad)
        (permanent if perm else transient).append((g, why))
    return ok, permanent, transient


def remove_raw(granules, raw_dir):
    for g in granules:
        for p in granule_files(g, raw_dir).values():
            for q in (p, p + ".part"):
                try:
                    os.remove(q)
                except OSError:
                    pass

In [ ]:
# ═══════════════ CELL 7: per-tile mask, dask filter task, write/skip rule ═══════════════
def layer_fill(dtype, nodata):
    """The value written outside the polygon: the source's own nodata when it declares one,
    otherwise NaN for floats / the type's extreme for integers. Returns (fill, declared)."""
    dt = np.dtype(dtype)
    if nodata is not None and not (isinstance(nodata, float) and math.isnan(nodata)):
        fill = dt.type(nodata)
        if float(fill) != float(nodata):
            raise DataIntegrityError(f"source nodata {nodata} does not fit dtype {dt}")
        return fill, True
    if dt.kind == "f":
        return dt.type(np.nan), nodata is not None
    info = np.iinfo(dt)
    return dt.type(info.max if dt.kind == "u" else info.min), False


def same_grid(a, b):
    return (tuple(a["shape"]) == tuple(b["shape"])
            and all(abs(x - y) <= 1e-6 for x, y in zip(a["transform"], b["transform"]))
            and rasterio.crs.CRS.from_wkt(a["crs_wkt"]) == rasterio.crs.CRS.from_wkt(b["crs_wkt"]))


def build_tile_ctx(tile, profiles):
    """Everything that is constant for one MGRS tile, built once from its first verified
    granule: native grid, the boolean polygon mask and the window that bounds it.
    Returns None when no pixel centre of the tile lies inside the polygon."""
    p = profiles[PRIMARY]
    for l in LAYERS:
        if not same_grid(p, profiles[l]):
            raise DataIntegrityError(f"{tile}: layer {l} is not on the grid of {PRIMARY}")
    h, w = p["shape"]
    transform = rasterio.Affine(*p["transform"])
    roi_native = gpd.GeoSeries([ROI_EXACT], crs=4326).to_crs(p["crs_wkt"]).iloc[0]
    # GDAL rasterisation in C: True where the pixel centre is inside the polygon
    full = rasterio.features.geometry_mask([roi_native], out_shape=(h, w), transform=transform,
                                           invert=True, all_touched=False)
    if not full.any():
        return None
    rows = np.flatnonzero(full.any(axis=1))
    cols = np.flatnonzero(full.any(axis=0))
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    layers = {}
    for l in LAYERS:
        fill, declared = layer_fill(profiles[l]["dtype"], profiles[l]["nodata"])
        layers[l] = dict(dtype=profiles[l]["dtype"], fill=fill, nodata_declared=declared,
                         tags=profiles[l].get("tags", {}))
    mask = np.ascontiguousarray(full[r0:r1, c0:c1])
    return dict(tile=tile, crs_wkt=p["crs_wkt"], transform=tuple(p["transform"]), shape=(h, w),
                win=(r0, r1, c0, c1), mask=mask, n_roi=int(mask.sum()), roi_frac=float(full.mean()),
                layers=layers, order=list(LAYERS), primary=PRIMARY)


def filter_granule(job, ctx):
    """ROI filter of one acquisition. Pure and deterministic, never raises; runs on a dask worker.

    Each layer is read in its native dtype for the window that bounds the polygon, then one
    vectorised np.where() sets pixels outside the polygon to the fill value. Nothing else
    touches the data: no cast, scale, clip, resample or QA masking.

    Returns {'status': 'ok', 'arrays': {layer: 2-D array}, 'stats': {...}} or a status of
    'grid_mismatch' / 'error' with a reason.
    """
    import numpy as np
    import rasterio
    from rasterio.windows import Window
    res = {"status": "error", "ur": job["ur"], "reason": ""}
    try:
        r0, r1, c0, c1 = ctx["win"]
        win = Window(c0, r0, c1 - c0, r1 - r0)
        mask = ctx["mask"]
        ref_crs = rasterio.crs.CRS.from_wkt(ctx["crs_wkt"])
        arrays = {}
        for layer in ctx["order"]:
            spec = ctx["layers"][layer]
            with rasterio.open(job["files"][layer]) as src:
                if ((src.height, src.width) != tuple(ctx["shape"]) or src.crs != ref_crs
                        or any(abs(a - b) > 1e-6 for a, b in zip(tuple(src.transform)[:6], ctx["transform"]))):
                    res.update(status="grid_mismatch", reason=f"{layer}: grid differs from the tile's reference grid")
                    return res
                arr = src.read(1, window=win)
            if str(arr.dtype) != spec["dtype"]:
                res.update(status="grid_mismatch", reason=f"{layer}: dtype {arr.dtype}, expected {spec['dtype']}")
                return res
            arrays[layer] = np.where(mask, arr, spec["fill"])
            del arr
        n_roi = int(ctx["n_roi"])
        prim = arrays[ctx["primary"]]
        fill = ctx["layers"][ctx["primary"]]["fill"]
        if prim.dtype.kind == "f":
            valid = ~np.isnan(prim)
            if not np.isnan(fill):
                valid &= prim != fill
        else:
            valid = prim != fill
        n_valid = int(np.count_nonzero(valid))
        stats = {"n_roi": n_roi, "n_valid": n_valid, "valid_frac": n_valid / n_roi}
        for layer in ctx["order"]:
            if layer != ctx["primary"] and arrays[layer].dtype.kind in "iu":
                stats[layer + "_frac"] = int(np.count_nonzero(arrays[layer] == 1)) / n_roi
        res.update(status="ok", arrays=arrays, stats=stats)
    except Exception as e:
        res.update(status="error", reason=f"{type(e).__name__}: {str(e)[:200]}")
    return res


def should_write(stats):
    """Decide whether one acquisition becomes a record in the tile's .nc (True) or is only
    logged to skipped_granules.jsonl (False). Called once per verified acquisition.

    stats (all refer to the part of this tile that lies inside the aquifer polygon):
      n_roi       number of 70 m pixels inside the polygon
      n_valid     how many of them hold a real ETdaily value in this acquisition
      valid_frac  n_valid / n_roi   (0.0 .. 1.0)
      cloud_frac  share of the in-polygon pixels flagged cloud
      water_frac  share of the in-polygon pixels flagged water
    MIN_VALID_FRAC (Cell 2, default 0.05) is available as the threshold.
    """
    # TODO(human): replace the placeholder below with your rule.
    return stats["n_valid"] > 0

In [ ]:
# ═══════════════ CELL 8: CF NetCDF writer, resume state, bit-exact check, tile audit ═══════════════
def tile_filename(tile):
    return f"{PRODUCT_TAG}_{tile}_{ROI_LABEL}.nc"


def _cf_attr(v):
    """netCDF attributes must be text or numbers."""
    if isinstance(v, (str, int, float, np.number, np.ndarray)):
        return v
    if isinstance(v, (list, tuple)):
        return np.asarray(v)
    return str(v)


def create_tile_nc(path, ctx):
    """Create the empty CF file of one tile on its native UTM grid (cropped to the window
    that bounds the polygon). Written under a temporary name and renamed when complete."""
    r0, r1, c0, c1 = ctx["win"]
    ny, nx = r1 - r0, c1 - c0
    a, b, c, d, e, f = ctx["transform"]
    crs = pyproj.CRS.from_wkt(ctx["crs_wkt"])
    xs = c + (c0 + np.arange(nx) + 0.5) * a                 # pixel-centre coordinates
    ys = f + (r0 + np.arange(ny) + 0.5) * e
    X, Y = np.meshgrid(xs, ys)
    lon, lat = pyproj.Transformer.from_crs(crs, 4326, always_xy=True).transform(X, Y)
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    tmp = path + ".new"
    ds = nc4.Dataset(tmp, "w", format="NETCDF4")
    try:
        ds.createDimension("time", None)
        ds.createDimension("y", ny)
        ds.createDimension("x", nx)

        tv = ds.createVariable("time", "f8", ("time",), chunksizes=(512,))
        tv.units, tv.calendar, tv.standard_name, tv.axis = TIME_UNITS, TIME_CALENDAR, "time", "T"
        tv.long_name = "acquisition start time (UTC), taken from the granule name"

        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis = "projection_y_coordinate", "m", "Y"
        yv.long_name = "y coordinate of pixel centre in the tile's UTM projection"
        yv[:] = ys
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis = "projection_x_coordinate", "m", "X"
        xv.long_name = "x coordinate of pixel centre in the tile's UTM projection"
        xv[:] = xs

        coords = None
        if WRITE_LATLON:
            la = ds.createVariable("lat", "f8", ("y", "x"), **zkw)
            la.standard_name, la.units, la.long_name = "latitude", "degrees_north", "latitude of pixel centre (WGS 84)"
            la[:] = lat
            lo = ds.createVariable("lon", "f8", ("y", "x"), **zkw)
            lo.standard_name, lo.units, lo.long_name = "longitude", "degrees_east", "longitude of pixel centre (WGS 84)"
            lo[:] = lon
            coords = "lat lon"

        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, _cf_attr(v))
        gm.spatial_ref = ctx["crs_wkt"]
        gm.GeoTransform = f"{c + c0 * a} {a} {b} {f + r0 * e} {d} {e}"     # GDAL order, this subset
        gm.tile_transform = np.asarray(ctx["transform"], dtype="f8")       # full tile, affine a..f
        gm.tile_shape = np.asarray(ctx["shape"], dtype="i4")               # full tile rows, cols
        gm.tile_window = np.asarray(ctx["win"], dtype="i4")                # row0, row1, col0, col1 stored here
        gm.long_name = "native grid of the MGRS tile (UTM, 70 m)"

        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), **zkw)
        mv.long_name = f"1 where the pixel centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values = np.array([0, 1], dtype="i1")
        mv.flag_meanings = "outside_roi inside_roi"
        mv.grid_mapping = "crs"
        if coords:
            mv.coordinates = coords
        mv[:] = ctx["mask"].astype("i1")

        chunks = (1, min(CHUNK_YX, ny), min(CHUNK_YX, nx))
        for layer in LAYERS:
            spec, meta = ctx["layers"][layer], LAYER_META.get(layer, {})
            dt = np.dtype(spec["dtype"])
            v = ds.createVariable(layer, dt, ("time", "y", "x"), fill_value=spec["fill"],
                                  chunksizes=chunks, **zkw)
            v.long_name = meta.get("long_name", f"{layer} layer of {SHORT_NAME}")
            v.units = meta.get("units", "1")
            v.grid_mapping = "crs"
            if coords:
                v.coordinates = coords
            if "flag_values" in meta:
                v.flag_values = np.array(meta["flag_values"], dtype=dt)
                v.flag_meanings = meta["flag_meanings"]
            if layer in SCIENCE_LAYERS and QA_LAYERS:
                v.ancillary_variables = " ".join(QA_LAYERS)
            v.source_dtype = spec["dtype"]
            v.source_nodata_declared = np.int8(bool(spec["nodata_declared"]))
            v.source_tags = json.dumps(spec.get("tags", {}), sort_keys=True, default=str)
            v.comment = ("Values inside the polygon are bit-identical to the source GeoTIFF. Pixels "
                         "outside the polygon are _FillValue. No scaling, clipping, resampling or "
                         "quality masking has been applied."
                         + (" Cloud and water flags are stored separately and are NOT applied."
                            if layer in SCIENCE_LAYERS else "")
                         + ("" if spec["nodata_declared"] else " The source declares no nodata value; "
                            "_FillValue here marks only pixels outside the polygon."))

        ov = ds.createVariable("orbit", "i4", ("time",), chunksizes=(512,))
        ov.long_name = "ISS orbit number of the acquisition"
        sv = ds.createVariable("scene", "i4", ("time",), chunksizes=(512,))
        sv.long_name = "scene number within the orbit"
        fv = ds.createVariable("roi_valid_fraction", "f4", ("time",), chunksizes=(512,))
        fv.long_name = f"share of in-polygon pixels of this tile holding a valid {PRIMARY} value"
        fv.units = "1"
        gv = ds.createVariable("source_granule", str, ("time",))
        gv.long_name = "name of the source granule of each record"

        inside = ctx["mask"]
        ds.Conventions = "CF-1.9"                      # 1.9: first version that admits unsigned integers
        ds.title = f"{SHORT_NAME} v{VERSION} - {ROI_LABEL} ROI subset, MGRS tile {ctx['tile']}, native 70 m grid"
        ds.summary = ("Per-acquisition stack of ECOSTRESS tiled layers for one MGRS tile on its native "
                      "UTM grid, cropped to the bounding window of the aquifer polygon. Source values "
                      "are preserved bit-for-bit; only pixels outside the polygon are set to _FillValue.")
        ds.institution = "NASA Jet Propulsion Laboratory / LP DAAC"
        ds.source = f"NASA ECOSTRESS {SHORT_NAME} v{VERSION} (LP DAAC), retrieved with earthaccess"
        ds.references = PRODUCT_DOI
        ds.history = f"{iso(utcnow())} created by ECOSTRESS_JET_Ogallala.ipynb"
        ds.product_short_name, ds.product_version, ds.mgrs_tile, ds.roi = SHORT_NAME, VERSION, ctx["tile"], ROI_LABEL
        ds.layers = " ".join(LAYERS)
        ds.tile_roi_fraction = np.float64(ctx["roi_frac"])
        ds.geospatial_lat_min, ds.geospatial_lat_max = float(lat[inside].min()), float(lat[inside].max())
        ds.geospatial_lon_min, ds.geospatial_lon_max = float(lon[inside].min()), float(lon[inside].max())
        ds.processing_status = "in_progress"
        ds.resume_after = ""
        ds.committed_records = np.int32(0)
    finally:
        ds.close()
    os.replace(tmp, path)
    log.info("[%s] created %s: window %d x %d of %d x %d, %d px inside the polygon (%.1f %% of the tile)",
             ctx["tile"], os.path.basename(path), ny, nx, ctx["shape"][0], ctx["shape"][1],
             ctx["n_roi"], 100 * ctx["roi_frac"])


def ctx_from_nc(path):
    """Rebuild the tile context from the file itself, so a resume needs no raw file."""
    with nc4.Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        missing = [l for l in LAYERS if l not in ds.variables]
        if missing:
            raise PipelineStop(f"{os.path.basename(path)} holds layers [{getattr(ds, 'layers', '?')}] but the "
                               f"settings ask for {LAYERS}. Restore the settings or use a new output folder.")
        gm = ds.variables["crs"]
        layers = {}
        for l in LAYERS:
            v = ds.variables[l]
            layers[l] = dict(dtype=str(v.dtype), fill=np.dtype(v.dtype).type(v.getncattr("_FillValue")),
                             nodata_declared=bool(v.getncattr("source_nodata_declared")), tags={})
        mask = np.asarray(ds.variables["roi_mask"][:]).astype(bool)
        return dict(tile=str(ds.mgrs_tile), crs_wkt=str(gm.spatial_ref),
                    transform=tuple(float(x) for x in gm.tile_transform),
                    shape=tuple(int(x) for x in gm.tile_shape), win=tuple(int(x) for x in gm.tile_window),
                    mask=mask, n_roi=int(mask.sum()), roi_frac=float(ds.tile_roi_fraction),
                    layers=layers, order=list(LAYERS), primary=PRIMARY)


def read_tile_state(path):
    """Resume state of a tile file, or None when the file cannot be trusted (unreadable, a
    record count that disagrees with the last commit, or a time axis that is not increasing)."""
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            n = len(ds.variables["time"])
            committed = int(getattr(ds, "committed_records", -1))
            if committed != n:
                log.warning("%s: %d records but %d committed - torn write, not trusted",
                            os.path.basename(path), n, committed)
                return None
            t = np.asarray(ds.variables["time"][:], dtype="f8")
            if n > 1 and not np.all(np.diff(t) > 0):
                log.warning("%s: time axis is not strictly increasing - not trusted", os.path.basename(path))
                return None
            ra = str(getattr(ds, "resume_after", ""))
            return dict(n=n, status=str(getattr(ds, "processing_status", "in_progress")),
                        resume_after=parse_iso(ra) if ra else None, size=os.path.getsize(path))
    except Exception as e:
        log.warning("%s cannot be read (%s) - not trusted", os.path.basename(path), e)
        return None


class TileWriter:
    """Single writer of one tile file. Records are appended strictly in time order; commit()
    stores the resume point and flushes, so the file on disk is always self-describing."""

    def __init__(self, path, cursor=None):
        self.path, self.ds, self.cursor = path, None, cursor
        self.open()

    def open(self):
        self.ds = nc4.Dataset(self.path, "a")
        self.ds.set_auto_maskandscale(False)
        tv = self.ds.variables["time"]
        self.n = len(tv)
        self.last = float(tv[self.n - 1]) if self.n else -math.inf
        self.dirty = 0

    def append(self, g, res):
        """Write one acquisition. Returns False (nothing written) for a duplicate or
        out-of-order time, which would break the CF time coordinate."""
        tv = self.ds.variables["time"]
        tnum = float(nc4.date2num(g["t"], tv.units, tv.calendar))   # units read back from the file
        if tnum <= self.last:
            return False
        i = self.n
        for layer in LAYERS:
            self.ds.variables[layer][i, :, :] = res["arrays"][layer]
        self.ds.variables["orbit"][i] = g["orbit"]
        self.ds.variables["scene"][i] = g["scene"]
        self.ds.variables["roi_valid_fraction"][i] = res["stats"]["valid_frac"]
        self.ds.variables["source_granule"][i] = g["ur"]
        tv[i] = tnum
        self.n, self.last = i + 1, tnum
        self.dirty += 1
        return True

    def commit(self, status=None):
        ds = self.ds
        ds.resume_after = iso(self.cursor) if self.cursor else ""
        ds.committed_records = np.int32(self.n)
        if status:
            ds.processing_status = status
        if self.n:
            tv = ds.variables["time"]
            first, last = nc4.num2date([float(tv[0]), float(tv[self.n - 1])], tv.units, tv.calendar)
            ds.time_coverage_start = first.strftime("%Y-%m-%dT%H:%M:%SZ")
            ds.time_coverage_end = last.strftime("%Y-%m-%dT%H:%M:%SZ")
        ds.sync()
        self.dirty = 0

    def close(self):
        if self.ds is not None:
            self.ds.close()
            self.ds = None


def bitexact_check(writer, index, files, ctx):
    """Re-read record `index` from the .nc and compare it bitwise with the source GeoTIFFs:
    identical inside the polygon, fill outside. Proof that nothing was manipulated."""
    r0, r1, c0, c1 = ctx["win"]
    win, mask = Window(c0, r0, c1 - c0, r1 - r0), ctx["mask"]
    for layer in LAYERS:
        with rasterio.open(files[layer]) as src:
            raw = src.read(1, window=win)
        got = np.asarray(writer.ds.variables[layer][index, :, :])
        if got.dtype != raw.dtype:
            raise DataIntegrityError(f"{layer}: stored dtype {got.dtype} differs from source {raw.dtype}")
        bits = f"u{got.dtype.itemsize}"
        fill, outside = ctx["layers"][layer]["fill"], got[~mask]
        out_ok = bool(np.all(np.isnan(outside))) if (got.dtype.kind == "f" and np.isnan(fill)) \
            else bool(np.all(outside == fill))
        if not np.array_equal(got[mask].view(bits), raw[mask].view(bits)) or not out_ok:
            raise DataIntegrityError(f"{ctx['tile']} record {index} layer {layer}: stored values differ from "
                                     f"{os.path.basename(files[layer])}")


def audit_tile_nc(path):
    """Read-only structural audit of a tile file. Returns (ok, problems, info)."""
    problems, info = [], {}
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            if not str(getattr(ds, "Conventions", "")).startswith("CF-"):
                problems.append("no CF Conventions attribute")
            need = ["time", "y", "x", "crs", "roi_mask", "orbit", "scene", "source_granule"] + LAYERS
            missing = [v for v in need if v not in ds.variables]
            if missing:
                return False, [f"missing variables {missing}"], info
            n = len(ds.variables["time"])
            t = np.asarray(ds.variables["time"][:], dtype="f8")
            if int(getattr(ds, "committed_records", -1)) != n:
                problems.append("record count differs from committed_records")
            if n > 1 and not np.all(np.diff(t) > 0):
                problems.append("time axis not strictly increasing")
            mask = np.asarray(ds.variables["roi_mask"][:])
            if not set(np.unique(mask).tolist()) <= {0, 1} or not mask.any():
                problems.append("roi_mask is empty or holds values other than 0/1")
            inside = mask.astype(bool)
            for l in LAYERS:
                v = ds.variables[l]
                if v.dimensions != ("time", "y", "x"):
                    problems.append(f"{l}: unexpected dimensions {v.dimensions}")
                if getattr(v, "grid_mapping", None) != "crs":
                    problems.append(f"{l}: grid_mapping attribute missing")
                if hasattr(v, "valid_range") or hasattr(v, "scale_factor") or hasattr(v, "add_offset"):
                    problems.append(f"{l}: carries a packing/valid_range attribute")
            pv = ds.variables[PRIMARY]
            fill = pv.getncattr("_FillValue")
            for i in sorted({0, n // 2, n - 1}) if n else []:
                a = np.asarray(pv[i, :, :])
                outside = a[~inside]
                if a.dtype.kind == "f" and np.isnan(fill):
                    leak = np.count_nonzero(~np.isnan(outside))
                else:
                    leak = np.count_nonzero(outside != fill)
                if leak:
                    problems.append(f"{PRIMARY} record {i}: {leak} non-fill pixels outside the polygon")
            if n:
                d0, d1 = nc4.num2date([t[0], t[-1]], ds.variables["time"].units, ds.variables["time"].calendar)
                info.update(first=str(d0), last=str(d1))
            info.update(records=n, status=str(getattr(ds, "processing_status", "?")),
                        window=f"{mask.shape[0]}x{mask.shape[1]}", size=os.path.getsize(path))
    except Exception as e:
        problems.append(f"cannot be opened/read: {type(e).__name__}: {e}")
    return (not problems), problems, info

In [ ]:
# ═══════════════ CELL 9: dask cluster, tile-by-tile pipeline, hourly backup, run ═══════════════
_BACKUP = {"next": 0.0}


def start_cluster():
    dask.config.set({"temporary-directory": DASK_TMP})
    cluster = LocalCluster(n_workers=DASK_WORKERS, threads_per_worker=1, processes=True,
                           memory_limit=f"{WORKER_MEMORY_GB}GiB", dashboard_address=None,
                           silence_logs=logging.WARNING)
    client = Client(cluster)
    log.info("Dask: %d worker processes x 1 thread, %.1f GiB memory limit each", DASK_WORKERS, WORKER_MEMORY_GB)
    return cluster, client


def _trim_worker_memory():
    import gc, sys, ctypes
    gc.collect()
    if sys.platform.startswith("linux"):
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)     # hand freed pages back to the OS
        except Exception:
            pass
    return True


def sync_workers(client):
    """Between download groups: every worker collects garbage and returns freed memory, so
    resident memory cannot creep up over thousands of acquisitions."""
    try:
        client.run(_trim_worker_memory)
    except Exception as e:
        log.debug("worker sync skipped: %s", e)
    gc.collect()


def backup_if_due(writer, drive_nc, force=False):
    """Wall-clock (hourly) copy of the tile in progress to Drive. The file is committed and
    closed first, so the Drive copy is always complete, consistent and resumable."""
    if not force and time.time() < _BACKUP["next"]:
        return False
    writer.commit()
    writer.close()
    try:
        robust_drive_copy(writer.path, drive_nc)
        _BACKUP["next"] = time.time() + BACKUP_INTERVAL_S
        return True
    except PipelineStop:
        raise
    except Exception as e:
        log.error("Backup to Drive failed (%s) - processing continues, next attempt in 10 min", e)
        _BACKUP["next"] = time.time() + 600
        return False
    finally:
        writer.open()


class Progress:
    def __init__(self, total):
        self.total, self.done, self.run, self.t0 = total, 0, 0, time.time()

    def add(self, n, resumed=False):
        self.done += n
        if not resumed:
            self.run += n

    def line(self):
        elapsed = max(time.time() - self.t0, 1e-9)
        rate = self.run / elapsed
        eta = (self.total - self.done) / rate / 3600 if rate > 0 else float("nan")
        return (f"{self.done}/{self.total} granules ({100 * self.done / max(1, self.total):.1f} %), "
                f"{rate * 60:.0f}/min, ETA {eta:.1f} h")


def filter_and_write(jobs, ctx, ctx_f, writer, client, drive_nc, checks_left):
    """Filter one group on the dask workers and append the results in time order.

    Memory stays flat: at most MAX_INFLIGHT results exist at any moment (running on workers
    or waiting here for their turn), each result is dropped as soon as it is written, and
    the .nc is flushed every SYNC_EVERY records.
    """
    counts = dict(written=0, skipped=0, failed=0)
    skipped_log, failed_log = [], []
    pending, ready = {}, {}
    pos = dict(submitted=0, nxt=0)
    ac = as_completed(loop=client.loop)

    def top_up():
        while pos["submitted"] < len(jobs) and len(pending) + len(ready) < MAX_INFLIGHT:
            i = pos["submitted"]
            fut = client.submit(filter_granule, dict(ur=jobs[i]["ur"], files=jobs[i]["files"]), ctx_f,
                                pure=False, retries=1)
            pending[fut] = i
            ac.add(fut)
            pos["submitted"] = i + 1

    def apply(job, res):
        nonlocal checks_left
        if res["status"] == "error":      # the file verified but its read failed: one retry in this process
            res = filter_granule(dict(ur=job["ur"], files=job["files"]), ctx)
        if res["status"] != "ok":
            counts["failed"] += 1
            failed_log.append(dict(ur=job["ur"], tile=ctx["tile"], reason=f"{res['status']}: {res['reason']}"))
        elif not should_write(res["stats"]):
            counts["skipped"] += 1
            skipped_log.append(dict(ur=job["ur"], tile=ctx["tile"], stage="write rule", **res["stats"]))
        elif writer.append(job, res):
            counts["written"] += 1
            if checks_left > 0:
                bitexact_check(writer, writer.n - 1, job["files"], ctx)
                checks_left -= 1
        else:
            counts["skipped"] += 1
            skipped_log.append(dict(ur=job["ur"], tile=ctx["tile"], stage="duplicate or out-of-order time"))
        writer.cursor = job["t"]

    top_up()
    for fut in ac:
        i = pending.pop(fut)
        try:
            res = fut.result()
        except Exception as e:
            res = dict(status="error", ur=jobs[i]["ur"], reason=f"task failed: {type(e).__name__}: {e}")
        del fut                                   # releases the result on the worker
        ready[i] = res
        del res
        while pos["nxt"] in ready:
            k = pos["nxt"]
            apply(jobs[k], ready.pop(k))
            pos["nxt"] = k + 1
            if writer.dirty >= SYNC_EVERY:
                writer.commit()                   # flush to disk and record the resume point
                backup_if_due(writer, drive_nc)
        top_up()
    log_events(LOG_SKIPPED, skipped_log)
    log_events(LOG_FAILED, failed_log)
    return counts, checks_left


def process_tile(T, client, prog):
    """One MGRS tile from its resume point to the end of the record."""
    tile = T["tile"]
    local_nc = os.path.join(NC_DIR, tile_filename(tile))
    drive_nc = os.path.join(DRIVE_OUT, tile_filename(tile))
    raw_dir = os.path.join(RAW_DIR, tile)
    granules = T["granules"][:TEST_GRANULES] if TEST_MODE else T["granules"]
    forced_skip = set(SKIP_GRANULES)
    t_start = time.time()

    # ---------- resume: find the newest trustworthy copy of this tile ----------
    lstate = read_tile_state(local_nc) if os.path.exists(local_nc) else None
    dstate = read_tile_state(drive_nc) if os.path.exists(drive_nc) else None
    if os.path.exists(drive_nc) and dstate is None:
        aside = f"{drive_nc}.unreadable_{utcnow():%Y%m%dT%H%M%S}"
        os.replace(drive_nc, aside)
        log.error("[%s] the Drive copy could not be trusted - moved aside as %s", tile, os.path.basename(aside))
    if lstate is not None and (dstate is None or lstate["n"] >= dstate["n"]):
        state = lstate
        log.info("[%s] continuing from the scratch copy", tile)
    elif dstate is not None:
        after = dstate["resume_after"]
        if dstate["status"] == "complete" and not any(after is None or g["t"] > after for g in granules):
            prog.add(len(granules), resumed=True)
            log.info("[%s] complete on Drive (%d records, %s) - skipped", tile, dstate["n"], human(dstate["size"]))
            return dict(tile=tile, status="already complete", records=dstate["n"], size=dstate["size"])
        restore_from_drive(drive_nc, local_nc)
        state = read_tile_state(local_nc)
        if state is None:
            raise PipelineStop(f"[{tile}] the copy restored from Drive is unreadable - re-run to retry")
    else:
        state = None
        if os.path.exists(local_nc):
            os.remove(local_nc)                      # untrusted leftover of a crashed kernel
    cursor = state["resume_after"] if state else None
    n_before = sum(1 for g in granules if cursor is not None and g["t"] <= cursor)
    prog.add(n_before, resumed=True)
    if state:
        log.info("[%s] RESUME: %d records in the file, last acquisition handled %s, %d granules still to do",
                 tile, state["n"], cursor, len(granules) - n_before)

    shutil.rmtree(raw_dir, ignore_errors=True)
    writer = ctx = ctx_f = header_bytes = None
    counts = dict(written=0, skipped=0, failed=0)
    checks_left = len(granules) if TEST_MODE else BITEXACT_CHECKS
    waits = 0
    try:
        if state is not None:
            ctx = ctx_from_nc(local_nc)
            writer = TileWriter(local_nc, cursor)
            if state["status"] != "in_progress":
                writer.commit(status="in_progress")   # a finished tile that has new acquisitions
            ctx_f = client.scatter([ctx], broadcast=True)[0]

        while True:
            todo = [g for g in granules if cursor is None or g["t"] > cursor][:GROUP_GRANULES]
            if not todo:
                break
            fetch = [g for g in todo if g["ur"] not in forced_skip]
            ensure_disk_free(f"{tile} download", extra_gb=sum(g["mb"] for g in fetch) / 1024)
            ok, permanent, transient = download_and_verify(fetch, raw_dir, client) if fetch else ([], [], [])
            permanent += [(g, "listed in SKIP_GRANULES") for g in todo if g["ur"] in forced_skip]
            # Nothing at or after the first unfetched granule may be committed: the resume
            # point only ever moves over acquisitions that are written, skipped or proven bad.
            cut = min(g["t"] for g, _ in transient) if transient else None
            ok = sorted((x for x in ok if cut is None or x[0]["t"] < cut), key=lambda x: x[0]["t"])
            permanent = [x for x in permanent if cut is None or x[0]["t"] < cut]

            if ctx is None and ok:
                t_setup = time.time()
                ctx = build_tile_ctx(tile, ok[0][2])
                if ctx is None:
                    log.warning("[%s] no pixel centre inside the polygon - tile dropped", tile)
                    shutil.rmtree(raw_dir, ignore_errors=True)
                    prog.add(len(granules) - n_before)
                    return dict(tile=tile, status="outside polygon")
                create_tile_nc(local_nc, ctx)
                header_bytes = os.path.getsize(local_nc)      # grid, lat/lon, mask: fixed cost per tile
                writer = TileWriter(local_nc, cursor)
                ctx_f = client.scatter([ctx], broadcast=True)[0]
                TIMES["tile setup"] = TIMES.get("tile setup", 0.0) + time.time() - t_setup
            if ok:
                jobs = [dict(ur=g["ur"], t=g["t"], orbit=g["orbit"], scene=g["scene"], files=files)
                        for g, files, _ in ok]
                t_fw, d0 = time.time(), TIMES.get("Drive copy", 0.0)
                c, checks_left = filter_and_write(jobs, ctx, ctx_f, writer, client, drive_nc, checks_left)
                TIMES["filter + write"] = (TIMES.get("filter + write", 0.0) + time.time() - t_fw
                                           - (TIMES.get("Drive copy", 0.0) - d0))
                for k in counts:
                    counts[k] += c[k]
            if permanent:
                counts["failed"] += len(permanent)
                log_events(LOG_FAILED, [dict(ur=g["ur"], tile=tile, reason=why) for g, why in permanent])
                for g, why in permanent:
                    log.warning("[%s] permanently unavailable: %s (%s)", tile, g["ur"], why[:160])
            handled = [x[0] for x in ok] + [x[0] for x in permanent]
            if handled:
                cursor = max(g["t"] for g in handled)
                if writer is not None:
                    writer.cursor = cursor
                    writer.commit()
                remove_raw(handled, raw_dir)
                prog.add(len(handled))
                STALL.progress()
                waits = 0
                log.info("[%s] through %s | written %d, skipped %d, failed %d | %s | free %.0f GB",
                         tile, cursor.strftime("%Y-%m-%d"), counts["written"], counts["skipped"],
                         counts["failed"], prog.line(), free_gb())
            if writer is not None:
                backup_if_due(writer, drive_nc)
            sync_workers(client)
            if transient:
                g0, why0 = min(transient, key=lambda x: x[0]["t"])
                log.warning("[%s] %d granule(s) could not be fetched; first: %s (%s)",
                            tile, len(transient), g0["ur"], why0[:200])
                if STALL.failure():
                    raise PipelineStop(
                        f"No progress for {STALL_MINUTES} min; stuck at {g0['ur']} ({why0[:200]}). "
                        "Everything so far is saved - re-run to resume. If this same granule blocks every "
                        "run while the server is otherwise fine, add its name to SKIP_GRANULES.")
                waits += 1
                time.sleep(min(30 * 2 ** waits, 600))

        # ---------- end of the record for this tile ----------
        if writer is None:
            log.warning("[%s] no usable acquisition - no file written", tile)
            return dict(tile=tile, status="no data", **counts)
        writer.cursor = cursor
        writer.commit()
        writer.close()
        good, problems, info = audit_tile_nc(local_nc)
        if good:
            writer.open()
            writer.commit(status="complete")
            writer.close()
        else:
            log.error("[%s] AUDIT FAILED - uploaded as in_progress: %s", tile, problems)
        robust_drive_copy(local_nc, drive_nc)
        size = os.path.getsize(local_nc)
        os.remove(local_nc)
        shutil.rmtree(raw_dir, ignore_errors=True)
        summary = dict(tile=tile, status="complete" if good else "audit failed", records=info.get("records", 0),
                       size=size, header_bytes=header_bytes, seconds=round(time.time() - t_start),
                       problems=problems, **counts)
        log_events(LOG_TILES, [summary])
        log.info("[%s] DONE: %d records, %s on Drive, %d skipped, %d failed, %.0f min",
                 tile, summary["records"], human(size), counts["skipped"], counts["failed"],
                 (time.time() - t_start) / 60)
        return summary
    except (PipelineStop, KeyboardInterrupt):
        # clean stop: leave Drive holding the newest consistent copy of this tile
        if writer is not None:
            try:
                if writer.ds is None:
                    writer.open()
                writer.commit()
                writer.close()
                if free_gb() >= MIN_FREE_GB:
                    robust_drive_copy(local_nc, drive_nc)
            except Exception as e:
                log.error("[%s] final checkpoint failed: %s", tile, e)
        raise
    finally:
        if writer is not None:
            try:
                writer.close()
            except Exception:
                pass
        del ctx_f


def preflight(client, tiles):
    """Fail fast before any bulk work: Drive is writable, the disk floor holds, and one real
    acquisition downloads and decodes. Logs how the source rasters describe themselves."""
    local, remote = os.path.join(NC_DIR, "_write_test.txt"), os.path.join(DRIVE_OUT, "_write_test.txt")
    with open(local, "w") as f:
        f.write(iso(utcnow()))
    robust_drive_copy(local, remote)
    os.remove(remote)
    os.remove(local)
    ensure_disk_free("preflight")
    raw = os.path.join(RAW_DIR, "_preflight")
    candidates = [g for T in tiles for g in T["granules"][:2] if g["ur"] not in SKIP_GRANULES][:3]
    why = "no candidate granule"
    try:
        for g in candidates:
            shutil.rmtree(raw, ignore_errors=True)
            ok, permanent, transient = download_and_verify([g], raw, client)
            if not ok:
                why = (permanent or transient)[0][1]
                continue
            _, _, prof = ok[0]
            p = prof[PRIMARY]
            log.info("PREFLIGHT OK with %s", g["ur"])
            log.info("   grid: %s, %d x %d px, pixel %.1f m", pyproj.CRS.from_wkt(p["crs_wkt"]).name,
                     p["shape"][0], p["shape"][1], p["transform"][0])
            for l in LAYERS:
                log.info("   %-10s dtype=%-8s nodata=%-6s %s  tags=%s", l, prof[l]["dtype"], prof[l]["nodata"],
                         human(prof[l]["size"]), prof[l]["tags"])
            return
        raise PipelineStop(
            "PREFLIGHT FAILED: " + why[:300] + "\n  1. Colab Secrets EARTHDATA_USERNAME / EARTHDATA_PASSWORD "
            "present and notebook access allowed?\n  2. Log in once at https://urs.earthdata.nasa.gov and accept "
            "any pending agreements.\n  3. Restart the runtime and re-run Cell 1 (pinned fsspec).\n"
            "  4. Check space: !df -h /content")
    finally:
        shutil.rmtree(raw, ignore_errors=True)


def mission_hits():
    try:
        return int(earthaccess.granule_query().short_name(SHORT_NAME).version(VERSION).polygon(ROI_RING).hits())
    except Exception as e:
        log.warning("Could not count the whole-mission granules: %s", e)
        return None


def run_pipeline():
    t0 = time.time()
    TIMES.clear()
    log.info("=" * 78)
    log.info("  %s v%s | layers %s | TEST_MODE=%s", SHORT_NAME, VERSION, LAYERS, TEST_MODE)
    log.info("  output: %s", DRIVE_OUT)
    log.info("=" * 78)
    earthdata_login()
    watchdog = DiskWatchdog()
    watchdog.start()
    cluster = client = prog = None
    results, stopped, kept_share, n_kept_tiles = [], None, None, 0
    try:
        ensure_disk_free("start")
        recs = get_manifest()
        tiles = select_tiles(build_tile_index(recs))
        kept_share = sum(len(T["granules"]) for T in tiles) / max(1, len(recs))
        n_kept_tiles = len(tiles)
        if ONLY_TILES:
            wanted = set(ONLY_TILES)
            absent = wanted - {T["tile"] for T in tiles}
            if absent:
                log.warning("ONLY_TILES not available (unknown, or below the overlap threshold): %s", sorted(absent))
            tiles = [T for T in tiles if T["tile"] in wanted]
            log.info("ONLY_TILES: restricted to %s", [T["tile"] for T in tiles])
        if TEST_MODE and tiles:
            ranked = sorted(tiles, key=lambda T: (T["roi_frac"] if T["roi_frac"] == T["roi_frac"] else 0.0, T["tile"]))
            n_pick = min(TEST_TILES, len(ranked))
            tiles = [ranked[int((i + 0.5) * len(ranked) / n_pick)] for i in range(n_pick)]
        if not tiles:
            raise PipelineStop("No tile passes MIN_TILE_ROI_FRAC - nothing to do.")
        prog = Progress(sum(len(T["granules"][:TEST_GRANULES] if TEST_MODE else T["granules"]) for T in tiles))
        cluster, client = start_cluster()
        preflight(client, tiles)
        _BACKUP["next"] = time.time() + BACKUP_INTERVAL_S
        for k, T in enumerate(tiles, 1):
            log.info("-" * 78)
            log.info("TILE %d/%d  %s  (%.1f %% of the tile inside the polygon, %d granules listed)",
                     k, len(tiles), T["tile"], 100 * T["roi_frac"], len(T["granules"]))
            results.append(process_tile(T, client, prog))
    except PipelineStop as e:
        stopped = e
    finally:
        watchdog.stop()
        for obj in (client, cluster):
            try:
                if obj is not None:
                    obj.close()
            except Exception:
                pass

    done = [r for r in results if r.get("status") == "complete"]
    written = sum(r.get("written", 0) for r in done)
    size = sum(r.get("size", 0) for r in done)
    log.info("=" * 78)
    if stopped is not None:
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing is lost, re-run to resume)\n  {stopped}\n" + "=" * 78 + "\n")
    log.info("  tiles: %s", {s: sum(1 for r in results if r.get("status") == s)
                             for s in sorted({r.get("status") for r in results})})
    log.info("  this run: %d records written, %s uploaded, %.1f min", written, human(size), (time.time() - t0) / 60)
    if TIMES:
        total_t = sum(TIMES.values())
        log.info("  time by stage: %s", ", ".join(f"{k} {v:.0f} s ({100 * v / total_t:.0f} %)"
                                                   for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))
    fresh = [r for r in done if r.get("header_bytes")]
    if TEST_MODE and prog is not None and prog.run and fresh and sum(r["written"] for r in fresh):
        hits = mission_hits()
        if hits:
            n_all = hits * kept_share
            w = sum(r["written"] for r in fresh)
            per_record = sum(r["size"] - r["header_bytes"] for r in fresh) / w
            header = sum(r["header_bytes"] for r in fresh) / len(fresh)
            per_granule = sum(TIMES.get(k, 0.0) for k in ("download", "corruption check", "filter + write")) / prog.run
            per_tile = sum(TIMES.get(k, 0.0) for k in ("tile setup", "Drive copy")) / len(results)
            log.info("  PROJECTION for the whole mission (from this rehearsal, rough):")
            log.info("     ~%.0f granules in %d tiles >= %.0f %% overlap (of %d granules over the polygon)",
                     n_all, n_kept_tiles, 100 * MIN_TILE_ROI_FRAC, hits)
            log.info("     ~%.0f h of runtime: %.2f s per granule + %.0f s per tile",
                     (n_all * per_granule + n_kept_tiles * per_tile) / 3600, per_granule, per_tile)
            log.info("     ~%.0f GB on Drive: %s per written record (%.0f %% of granules written) + %s per tile",
                     (n_all * (written / prog.run) * per_record + n_kept_tiles * header) / 1024 ** 3,
                     human(per_record), 100 * written / prog.run, human(header))
    log.info("=" * 78)
    return results


RESULTS = run_pipeline()

In [ ]:
# ═══════════════ CELL 10: audit every tile file on Drive ═══════════════
def audit_all():
    paths = sorted(glob.glob(os.path.join(DRIVE_OUT, f"{PRODUCT_TAG}_*_{ROI_LABEL}.nc")))
    print(f"{len(paths)} tile files in {DRIVE_OUT}\n")
    print(f"{'file':44s} {'status':12s} {'records':>8s} {'size':>10s}  first .. last acquisition")
    total, bad = 0, 0
    for p in paths:
        ok, problems, info = audit_tile_nc(p)
        total += info.get("size", 0)
        print(f"{os.path.basename(p):44s} {info.get('status', '?'):12s} {info.get('records', 0):8d} "
              f"{human(info.get('size', 0)):>10s}  {info.get('first', '-')} .. {info.get('last', '-')}")
        if not ok:
            bad += 1
            for msg in problems:
                print(f"      PROBLEM: {msg}")
    print(f"\nTotal {human(total)} | {bad} file(s) with problems")
    for name in (LOG_FAILED, LOG_CORRUPT, LOG_SKIPPED):
        path = os.path.join(DRIVE_OUT, name)
        n = sum(1 for _ in open(path)) if os.path.exists(path) else 0
        print(f"{name}: {n} entries")


audit_all()

# Optional independent CF check of one tile file:
# !pip -q install compliance-checker
# !compliance-checker --test=cf:1.9 "/content/drive/MyDrive/MSUGWB/ECOSTRESS_ECO_L3T_JET_v002/<file>.nc"